# QBlade 几何级批量仿真

| 模式 | workers | 说明 |
|------|---------|------|
| GPU  | 1       | 单 GPU OpenCL (~17 min/geo) |
| CPU  | N       | N 进程并行, 每个独立 DLL + 文件树 |

断点续传: 自动读取 `geometry_results.pkl`, 按 `geometry_idx` 跳过已完成的几何

In [ ]:
import sys, os
import numpy as np
import pandas as pd

ROOT = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, ROOT)

from project_config import DLL_FILE, SIM_FOLDER, GEOMETRY_NPY
from get_data import run_geometry_batch, _load_checkpoint

geometry_list = np.load(GEOMETRY_NPY)
print(f'几何数量: {geometry_list.shape[0]}, 截面: {geometry_list.shape[1]}')

In [ ]:
DEVICE = 'GPU'
MAX_WORKERS = 1
OMP_THREADS = 4
RPM, WIND_SPEED, ANGLE = 6000, 10, 85
TIMESTEPS = 1800
OUTPUT_PKL = 'geometry_results.pkl'

existing = _load_checkpoint(OUTPUT_PKL)
completed = {r['geometry_idx'] for r in existing if r.get('error') is None}
pending = len(geometry_list) - len(completed)
print(f'已完成: {len(completed)}, 待执行: {pending}')

In [ ]:
results = run_geometry_batch(
    geometry_list,
    dll_file=DLL_FILE,
    sim_folder=SIM_FOLDER,
    rpm=RPM,
    wind_speed=WIND_SPEED,
    angle=ANGLE,
    num_timesteps=TIMESTEPS,
    device=DEVICE,
    max_workers=MAX_WORKERS,
    omp_threads=OMP_THREADS,
    output_pkl=OUTPUT_PKL,
    output_csv=OUTPUT_PKL.replace('.pkl', '.csv'),
)
print(f'完成: {len(results)} 条结果')

In [ ]:
ok = [r for r in results if r.get('error') is None]
err = [r for r in results if r.get('error') is not None]
print(f'成功: {len(ok)}, 失败: {len(err)}')

if ok:
    df = pd.DataFrame(ok)
    display(df[['geometry_idx', 'THRUST', 'POWER', 'Ct', 'Cp', 'eta']].describe())